In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
%run "../Modulo 4/Variaveis"

In [0]:
partidas_df = spark.read.parquet(f'{caminho_gold}fato_partidas')


In [0]:
df_reduzido = partidas_df.select('ano_torneio', 'id_time_casa', 'qt_gols_casa')

In [0]:
display(df_reduzido)

In [0]:
df_agrupado = df_reduzido.groupBy('ano_torneio', 'id_time_casa').sum('qt_gols_casa')

In [0]:
display(df_reduzido)

In [0]:
df_agrupado = df_reduzido.groupBy('ano_torneio', 'id_time_casa').sum('qt_gols_casa')

In [0]:
display(df_agrupado)

In [0]:
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import rank, dense_rank, row_number, desc

In [0]:
janela = Window.partitionBy('ano_torneio').orderBy(desc('sum(qt_gols_casa)'))

In [0]:
df_rank = df_agrupado.withColumn('rank', dense_rank().over(janela))

In [0]:
display(df_rank)

In [0]:
df_row_number = df_agrupado.withColumn('r_n', row_number().over(janela))

In [0]:
df_row_number.filter('r_n = 1').display()

In [0]:
from pyspark.sql.functions import col

In [0]:
df = spark.read.parquet(f'{caminho_gold}fato_partidas')


In [0]:
df_selecionado = df.select('ano_torneio', 'nr_partida', 'dt_partida', 'id_time_casa', 'id_time_fora', 'qt_gols_casa', 'qt_gols_fora')

In [0]:
display(df_selecionado)

In [0]:
df_total_gols = df_selecionado \
    .withColumn('total_gols', col('qt_gols_casa') + col('qt_gols_fora'))

In [0]:
display(df_total_gols)

In [0]:
times_df  = spark.read.parquet(f'{caminho_silver}times')

In [0]:
display(times_df)

In [0]:
janela = Window.partitionBy('ano_torneio').orderBy(col('total_gols').desc())

In [0]:
df_rank = df_total_gols.withColumn('rank', dense_rank().over(janela))

In [0]:
df_rank.display()

In [0]:
df_desafio = df_rank.join(
    times_df.alias('times_casa'),
    on=df_rank.id_time_casa == col('times_casa.id_time'),
    how='left'
).select(df_rank['*'], col('times_casa.nm_time').alias('time_casa'))

In [0]:
df_desafio = df_desafio.join(
    times_df.alias('times_fora'),
    on=df_rank.id_time_fora == col('times_fora.id_time'),
    how='left'
).select(df_desafio['*'], col('times_fora.nm_time').alias('time_fora'))    

In [0]:
display(df_desafio)

In [0]:
df_desafio.printSchema()

In [0]:
df_final = df_desafio.filter('ano_torneio = 2022 and rank = 1').select(
    'nr_partida',
    'dt_partida',
    'time_casa',
    'time_fora',
    'qt_gols_casa',
    'qt_gols_fora'
)

In [0]:
display(df_final)